## Database Setup and Connection Testing

This notebook sets up the IBM i database for 3SAT problems and tests the Mapepire connection.

**Author:** Quantum-IBMi-Docker Project  
**Based on:** Jack Woehr's COMMON 2021 Presentation

### Import Required Libraries

In [ ]:
import getpass
from mapepire_python.client.sql_job import SQLJob
from mapepire_python.data_types import DaemonServer
from IPython.display import Markdown, display

print("✓ All libraries imported successfully")

### Connect to IBM i via Mapepire

In [ ]:
# Get connection details from user
host = input("Enter IBM i hostname/IP: ").strip()
username = input("Enter username: ").strip()
password = getpass.getpass("Enter password: ")
port_input = input("Enter port (press Enter for default 8076): ").strip()
port = int(port_input) if port_input else 8076

# Create daemon server configuration
daemon_server = DaemonServer(
    host=host,
    port=port,
    user=username,
    password=password
)

# Create SQL job and connect
sql_job = SQLJob()
sql_job.connect(daemon_server)

display(Markdown("**✓ Connected to IBM i database**"))

### Test Connection with Simple Query

In [ ]:
# Test query to verify connection
print("Testing query: SELECT CURRENT_USER...")
query = sql_job.query("SELECT CURRENT_USER FROM SYSIBM.SYSDUMMY1")
result = query.run()

if result.get("success") and result.get("data"):
    first_row = result['data'][0]
    current_user = list(first_row.values())[0]
    print(f"✓ Current user: {current_user}")
else:
    print(f"✗ Query failed: {result.get('error', 'Unknown error')}")

### Create Database Schema

In [ ]:
# Create DIMACS schema if it doesn't exist
try:
    query = sql_job.query("CREATE SCHEMA DIMACS")
    result = query.run()
    if result.get("success"):
        print("✓ Schema DIMACS created successfully")
    else:
        print(f"Note: {result.get('error', 'Schema might already exist')}")
except Exception as e:
    print(f"Note: Couldn't create schema. Probably already exists. ({e})")

### Create 3SAT Table

In [ ]:
# Create the SAT3_1 table
create_table_sql = """
CREATE OR REPLACE TABLE DIMACS.SAT3_1 (
    A INTEGER,
    B INTEGER,
    C INTEGER
)
ON REPLACE DELETE ROWS
"""

try:
    query = sql_job.query(create_table_sql)
    result = query.run()
    if result.get("success"):
        print("✓ Table DIMACS.SAT3_1 created successfully")
    else:
        print(f"✗ Failed to create table: {result.get('error', 'Unknown error')}")
except Exception as e:
    print(f"✗ Error creating table: {e}")

### Insert Sample 3SAT Data

In [ ]:
# Insert sample 3SAT clauses
insert_data_sql = """
INSERT INTO DIMACS.SAT3_1 (A, B, C) VALUES
    (-1, -2, -3),
    (1, -2, 3),
    (1, 2, -3),
    (1, -2, -3),
    (-1, 2, 3)
"""

try:
    query = sql_job.query(insert_data_sql)
    result = query.run()
    if result.get("success"):
        print("✓ Sample 3SAT data inserted successfully")
        print("  5 clauses inserted into DIMACS.SAT3_1")
    else:
        print(f"✗ Failed to insert data: {result.get('error', 'Unknown error')}")
except Exception as e:
    print(f"✗ Error inserting data: {e}")

### Verify Data Insertion

In [ ]:
# Query the table to verify data
query = sql_job.query("SELECT * FROM DIMACS.SAT3_1")
result = query.run()

if result.get("success") and result.get("data"):
    print(f"✓ Data verification successful")
    print(f"  Found {len(result['data'])} rows in DIMACS.SAT3_1")
    print("\nSample data:")
    for i, row in enumerate(result['data'][:5], 1):
        print(f"  Row {i}: {row}")
else:
    print(f"✗ Failed to verify data: {result.get('error', 'No data found')}")

### Test THREESAT Function (if available)

In [ ]:
# Test the THREESAT function if it exists
print("Testing THREESAT function...")
try:
    query = sql_job.query("SELECT * FROM TABLE(JESSEG.THREESAT()) AS t")
    result = query.run()
    
    if result.get("success") and result.get("data"):
        print(f"✓ THREESAT function returned: {len(result['data'])} rows")
        print(f"\nSample data (first row): {result['data'][0]}")
        print(f"\nData structure: Each row represents a clause with literals")
        print(f"\nAll rows:")
        for i, row in enumerate(result['data'][:5], 1):
            print(f"  Row {i}: {row}")
    else:
        print(f"Note: THREESAT function not available or returned no data")
        print(f"  Error: {result.get('error', 'No data returned')}")
except Exception as e:
    print(f"Note: THREESAT function test skipped: {e}")

### Close Connection

In [ ]:
# Close the SQL job connection
sql_job.close()
print("✓ Connection closed successfully")

### Summary

This notebook has:
1. ✓ Connected to IBM i via Mapepire
2. ✓ Created the DIMACS schema
3. ✓ Created the SAT3_1 table
4. ✓ Inserted sample 3SAT data
5. ✓ Verified the data
6. ✓ Tested the THREESAT function (if available)

You can now run the `3sat.ipynb` notebook to solve 3SAT problems using Grover's algorithm!